# Chapter 10: Intro to Convolutional Neural Networks — Learning Edges and Corners
### Grokking Deep Learning (Andrew W. Trask)
**Student Name**: Muhammad Aidil Fadhli Awalludin  
**NIM**: 101032300087  
**Class**: TK4701  

---

> *"Vision begins with the detection of features: edges, corners, textures, and orientations."*  
> — David Marr, Vision (1982)

---

## 1. Executive Summary & Objective

In Chapter 7, we classified handwritten digits by flattening 2D images into 1D vectors and connecting them through dense (fully connected) layers. While functional for small $28 \times 28$ icons, dense networks suffer from fundamental architectural flaws when applied to real-world visual imagery.

### Core Objectives:
1. **Expose the Flaws of Fully Connected Vision Models**: Understand parameter explosion and the loss of translation invariance.
2. **Deconstruct the Convolution Operation**: Implement 2D kernel convolution and analyze weight sharing and local receptive fields.
3. **Implement Feature Pooling**: Construct Max Pooling layers to provide spatial downsampling and translational invariance.
4. **Build a Convolutional Neural Network from Scratch**: Code and train a working ConvNet with convolutional feature extractors and backpropagation using pure NumPy.

---
## 2. Theoretical Deep-Dive

### 2.1 Why Fully Connected Layers Fail on Vision

1. **Parameter Explosion**:
   - Consider a modern smartphone photo ($1000 \times 1000 \times 3 = 3,000,000$ inputs).
   - Connecting to a first hidden layer of 1,000 neurons requires $3 \times 10^9$ (3 billion) weights for a single layer, leading to memory exhaustion and instant overfitting.

2. **Lack of Translation Invariance**:
   - In a fully connected network, each pixel has a dedicated weight.
   - If a cat appears in the top-left corner, weights associated with top-left pixels learn cat features. If the same cat appears in the bottom-right corner, the network must learn cat features entirely from scratch!

3. **Destruction of 2D Topology**:
   - Flattening an image treats pixel $(1, 1)$ as equally adjacent to pixel $(1, 2)$ and pixel $(28, 28)$, destroying spatial locality.

### 2.2 The Convolution Operation & Weight Sharing

A **Convolutional Neural Network (CNN)** resolves these limitations using two core principles:
1. **Local Receptive Fields**: A neuron connects only to a small local patch of pixels (e.g., $3 \times 3$ or $5 \times 5$).
2. **Weight Sharing (Tied Weights)**: A single kernel (filter) $\mathbf{K}$ slides across the entire spatial grid, computing dot products at each position:

$$(\mathbf{I} * \mathbf{K})(i, j) = \sum_{m=0}^{k_h - 1} \sum_{n=0}^{k_w - 1} \mathbf{I}(i + m, j + n) \mathbf{K}(m, n)$$

```
Input Image (5x5)          Kernel (3x3)           Output Feature Map (3x3)
 [ 1  0  1  0  0 ]                                  [ 3  2  1 ]
 [ 0  1  1  1  0 ]    *     [ 1  0  1 ]      =      [ 2  4  2 ]
 [ 1  0  1  0  1 ]          [ 0  1  0 ]             [ 3  2  3 ]
 [ 0  1  0  1  0 ]          [ 1  0  1 ]   
 [ 1  0  1  0  1 ]
```

#### Output Dimension Formula:
Given input spatial dimension $W$, filter size $K$, padding $P$, and stride $S$:
$$O = \left\lfloor \frac{W - K + 2P}{S} \right\rfloor + 1$$

### 2.3 Max Pooling & Hierarchical Representations
- **Max Pooling**: Selects the maximum value within non-overlapping windows (e.g., $2 \times 2$ with stride 2). This cuts spatial area by $75\%$, retains dominant features, and introduces translational invariance.
- **Feature Hierarchy**: Early conv layers learn edges, lines, and textures; intermediate layers assemble edges into motifs and parts; deep layers recognize complex objects (faces, cars, digits).

---
## 3. Code Reproduction

We now implement 2D convolution and a complete mini-CNN pipeline from scratch using NumPy.

In [ ]:
# 3.1 2D Convolution Operation from Scratch
import numpy as np

def conv2d_forward(image, kernel, stride=1):
    H, W = image.shape
    kH, kW = kernel.shape
    out_H = (H - kH) // stride + 1
    out_W = (W - kW) // stride + 1
    
    output = np.zeros((out_H, out_W))
    for i in range(out_H):
        for j in range(out_W):
            patch = image[i*stride : i*stride + kH, j*stride : j*stride + kW]
            output[i, j] = np.sum(patch * kernel)
            
    return output

# Demonstration with edge detection kernels
sample_image = np.array([
    [0, 0, 10, 10, 0],
    [0, 0, 10, 10, 0],
    [0, 0, 10, 10, 0],
    [0, 0, 10, 10, 0],
    [0, 0, 10, 10, 0]
], dtype=float)

# Vertical edge detector (Sobel-like)
vertical_filter = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1]
], dtype=float)

edge_map = conv2d_forward(sample_image, vertical_filter)
print("Sample 5x5 Image with Vertical Stripe:\n", sample_image.astype(int))
print("\nDetected Vertical Edge Map (3x3):\n", np.round(edge_map, 1))

In [ ]:
# 3.2 Max Pooling Layer Implementation

def maxpool2d_forward(feature_map, pool_size=2, stride=2):
    H, W = feature_map.shape
    out_H = (H - pool_size) // stride + 1
    out_W = (W - pool_size) // stride + 1
    
    output = np.zeros((out_H, out_W))
    for i in range(out_H):
        for j in range(out_W):
            window = feature_map[i*stride : i*stride + pool_size, j*stride : j*stride + pool_size]
            output[i, j] = np.max(window)
            
    return output

sample_map = np.array([
    [1, 3, 2, 4],
    [5, 6, 7, 8],
    [3, 2, 1, 0],
    [1, 2, 3, 4]
])

pooled = maxpool2d_forward(sample_map, pool_size=2, stride=2)
print("Input Feature Map (4x4):\n", sample_map)
print("\nMax-Pooled Output (2x2):\n", pooled)

In [ ]:
# 3.3 Building & Training a Mini-ConvNet from Scratch

np.random.seed(42)

# Synthetic dataset: 8x8 images, 2 classes (Horizontal stripe vs Vertical stripe)
num_samples = 120
img_size = 8

X_data = np.zeros((num_samples, img_size, img_size))
y_data = np.zeros(num_samples)

for i in range(num_samples):
    if i % 2 == 0:
        # Vertical stripe class (y = 0)
        col = np.random.randint(1, img_size - 1)
        X_data[i, :, col] = 1.0
        y_data[i] = 0
    else:
        # Horizontal stripe class (y = 1)
        row = np.random.randint(1, img_size - 1)
        X_data[i, row, :] = 1.0
        y_data[i] = 1
    X_data[i] += np.random.normal(0, 0.05, (img_size, img_size))

# ConvNet Parameters: Single 3x3 filter, followed by dense output neuron
kernel = np.random.randn(3, 3) * 0.1
dense_weights = np.random.randn(36, 1) * 0.1  # (8-3+1)^2 = 6x6 = 36 flattened features
bias = 0.0
alpha = 0.05
epochs = 30

print("Training Mini-ConvNet for Pattern Classification:")
for epoch in range(epochs):
    total_loss = 0.0
    correct = 0
    
    for i in range(num_samples):
        img = X_data[i]
        target = y_data[i]
        
        # 1. Forward Pass (Convolution -> Flatten -> Linear -> Sigmoid)
        feat_map = conv2d_forward(img, kernel)
        # ReLU activation
        feat_relu = np.maximum(0, feat_map)
        flattened = feat_relu.reshape(1, -1)  # shape (1, 36)
        
        logit = np.dot(flattened, dense_weights) + bias
        pred = 1.0 / (1.0 + np.exp(-np.clip(logit, -100, 100)))
        
        # Loss and metric
        loss = (pred[0][0] - target) ** 2
        total_loss += loss
        if (pred[0][0] >= 0.5) == target:
            correct += 1
            
        # 2. Backward Pass
        delta = pred[0][0] - target
        d_dense = delta * flattened.T
        bias -= alpha * delta
        dense_weights -= alpha * d_dense
        
        # Backprop through flatten and ReLU to kernel
        d_flat = delta * dense_weights.T
        d_feat = (d_flat.reshape(6, 6)) * (feat_map > 0)
        
        # Gradient for shared kernel (accumulate across patches)
        d_kernel = np.zeros((3, 3))
        for r in range(6):
            for c in range(6):
                d_kernel += d_feat[r, c] * img[r:r+3, c:c+3]
                
        kernel -= alpha * d_kernel
        
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:2d} | Loss: {total_loss:7.4f} | Accuracy: {(correct/num_samples)*100:5.1f}%")

print("\nLearned 3x3 Convolutional Filter:\n", np.round(kernel, 4))

---
## 4. Chapter 10 Summary & Key Takeaways

| Concept | Architectural Feature | Advantage for Computer Vision |
|:---|:---:|:---|
| **Local Receptive Fields** | Small $k \times k$ patches | Preserves 2D spatial locality and prevents parameter explosion. |
| **Weight Sharing** | Shared kernel $\mathbf{K}$ | Imparts translational invariance; a detector operates identically anywhere in the image. |
| **Max Pooling** | Downsampling windows | Reduces computation and introduces spatial robustness to minor shifts. |
| **Gradient Accumulation** | $\Delta \mathbf{K} = \sum_{patch} \delta \cdot \text{patch}$ | The shared kernel accumulates gradients from all receptive fields across the entire image. |